# Klydo Scraper Debug Notebook

Use this to sanity-check the Klydo scraper locally (search + PDP).

Prereqs:
- Run inside this repo root.
- Ensure `.env` has `KLYDO_DEFAULT_SCRAPER=klydo` (optional but recommended) and a valid token if you have one (`KLYDO_KLYDO_API_TOKEN`).
- Kernel should point to the project venv (e.g., `.venv`).


In [1]:
import asyncio, json, os
from klydo.scrapers import get_scraper
from klydo.config import settings


In [2]:
# Quick visibility into the active env
print({
    'default_scraper': settings.default_scraper,
    'klydo_api_token_set': bool(settings.klydo_api_token),
    'klydo_session_id': settings.klydo_session_id,
    'env_file': settings.model_config.get('env_file'),
})


{'default_scraper': 'klydo', 'klydo_api_token_set': True, 'klydo_session_id': '1765794693441-605512', 'env_file': '.env'}


In [3]:
def pretty(label, payload):
    print(f"\n{label}:")
    print(json.dumps(payload, indent=2))

async def run_search(scraper, query: str, limit: int = 10):
    results = await scraper.search(query, limit=limit)
    print(f"Found {len(results)} results for '{query}'")
    for p in results:
        print(f"- {p.id} | {p.brand} | {p.name} | Rs {p.price.current}")
    return results

async def run_pdp(scraper, product_ids):
    for pid in product_ids:
        prod = await scraper.get_product(pid)
        if prod:
            pretty(pid, prod.model_dump(mode='json'))
        else:
            print(f"\n{pid}: None (not found)")


In [4]:
SCRAPER_NAME = 'klydo'
SEARCH_QUERY = 'Color Capital Athletic Jacket'
PRODUCT_IDS = [
    'STL_781SXKGTCJI6UVXG8HH0',
    'STL_NTPNGOCNMCCLKR5RJ9RZ',
    'STL_YW4USEN0V3O39HB1HYJ9',
]

async def main():
    scraper = get_scraper(SCRAPER_NAME)
    try:
        await run_search(scraper, SEARCH_QUERY, limit=10)
        await run_pdp(scraper, PRODUCT_IDS)
    finally:
        await scraper.close()

await main()


Found 10 results for 'Color Capital Athletic Jacket'
- STL_781SXKGTCJI6UVXG8HH0 | Color Capital | Greyish-Blue SkinnyFit Athletic Jacket | Rs 890
- STL_YW4USEN0V3O39HB1HYJ9 | Color Capital | Navy Seamless Athletic Tennis Jacket | Rs 890
- STL_NTPNGOCNMCCLKR5RJ9RZ | Color Capital | Black Athletic Zip-Up Sport Jacket | Rs 890
- STL_KILEOHMXRE0G1B4VU4GW | Color Capital | Brown Knitted Sporty Jacket with Front Zip | Rs 890
- STL_FA2NHOLR7XY2ML8L3Z1E | MASCLN SASSAFRAS | Blue Color-Block Denim Shacket with Chest Pockets | Rs 989
- STL_VFQ3BWA9EX514T6T17EE | Tokyo Talkies | Tokyo Talkies Women Blue Solid Jacket | Rs 1249
- STL_1R4XF5OW1BV1N71W3RJX | Tokyo Talkies | Tokyo Talkies Women Olive Green Solid Denim Jacket | Rs 1299
- STL_QB4EZMPG61S34HPUKXPM | HIGHLANDER | HIGHLANDER Men Blue Solid Jacket | Rs 1399
- STL_20RQ5ZH0HCVO2N4LYFHS | Tokyo Talkies | Tokyo Talkies Women Blue Solid Denim Jacket | Rs 1149
- STL_H094H6R81JQ9Q0KINK1D | Tokyo Talkies | Tokyo Talkies Women Navy Blue Solid Denim 

In [5]:
# Optional: swap in other queries/IDs here
ALT_QUERY = 'T-Shirts'
ALT_IDS = ['STL_KLOIFTGI59HN8UC9YVJ8']

async def alt_run():
    scraper = get_scraper(SCRAPER_NAME)
    try:
        await run_search(scraper, ALT_QUERY, limit=5)
        await run_pdp(scraper, ALT_IDS)
    finally:
        await scraper.close()

# Uncomment to run
await alt_run()


Found 5 results for 'T-Shirts'
- STL_4JX3FGXTAOGLK4WJ248W | CULT | Cult Essential Cotton Pack of 2 T-shirts | Rs 949
- STL_OV5QW00Z6R8Q9FIM513H | Recast | Why are we here ? : Men's Relaxed fit Maroon T-Shirts | Rs 399
- STL_8IW47D90UG06Y0V2SGL7 | Recast | You mad ? Stay Mad : Men's Relaxed fit Aqua Blue T-Shirts | Rs 399
- STL_OGEZTGB4504VF645RANM | Recast | I never finish anythin : Men's Relaxed fit Aqua Blue T-Shirts | Rs 399
- STL_E9B1LSKB2FQV7BI1IIRN | Recast | Enjoy, Don't Destroy : Men's Relaxed fit Aqua Blue T-Shirts | Rs 399

STL_KLOIFTGI59HN8UC9YVJ8: None (not found)
